# SmartDAM — exploration des modèles HuggingFace & investigation du bug "bird"

Carnet de travail (pas un tutoriel nettoyé) : exploration des sorties brutes de
ResNet-50 (classification) et DETR (détection d'objets) via l'API d'inférence
Hugging Face utilisée par `services/huggingface.py`, itérations sur les seuils
de confiance, construction du dictionnaire de traduction FR, et diagnostic du
bug où le tag "bird" apparaissait sur presque toutes les images.

Certaines cellules montrent des essais qui n'ont pas marché du premier coup —
c'est volontaire, ça documente la vraie démarche.

In [1]:
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from dotenv import load_dotenv
load_dotenv()

import requests

HF_API_BASE_URL = "https://router.huggingface.co/hf-inference/models"
HF_TOKEN = os.getenv("HUGGINGFACE_API_TOKEN", "")
CLASSIFICATION_MODEL = "microsoft/resnet-50"
DETECTION_MODEL = "facebook/detr-resnet-50"

print("Token configuré :", bool(HF_TOKEN))

DEMO_DIR = Path("../demo_assets") if Path("../demo_assets").exists() else Path("demo_assets")
demo_images = sorted(DEMO_DIR.glob("*.png"))
demo_images

Token configuré : True


[WindowsPath('../demo_assets/chef_station_portrait.png'),
 WindowsPath('../demo_assets/dessert_square_showcase.png'),
 WindowsPath('../demo_assets/fruit_market_table.png'),
 WindowsPath('../demo_assets/grilled_chicken_plate.png')]

In [2]:
def query_model(model_id: str, image_bytes: bytes, content_type: str = "image/png", timeout: int = 30):
    """Same call shape as HuggingFaceService._query_model, kept standalone here
    so this notebook doesn't depend on a running Flask app context."""
    response = requests.post(
        f"{HF_API_BASE_URL}/{model_id}",
        headers={
            "Authorization": f"Bearer {HF_TOKEN}",
            "Accept": "application/json",
            "Content-Type": content_type,
        },
        data=image_bytes,
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()

# Quick sanity call on the first demo image
sample_path = demo_images[0]
sample_bytes = sample_path.read_bytes()
print("Testing on:", sample_path.name, f"({len(sample_bytes)} bytes)")

Testing on: chef_station_portrait.png (23244 bytes)


## 1. Classification (ResNet-50) — sorties brutes

In [3]:
classification_results = {}
for path in demo_images:
    try:
        payload = query_model(CLASSIFICATION_MODEL, path.read_bytes())
        classification_results[path.name] = payload
        print(f"--- {path.name} ---")
        for item in payload[:5]:
            print(f"  {item['score']:.3f}  {item['label']}")
    except requests.exceptions.RequestException as exc:
        print(f"--- {path.name}: ERREUR ({exc}) ---")
        classification_results[path.name] = None

--- chef_station_portrait.png ---
  0.126  abaya
  0.045  bulletproof vest
  0.042  church, church building
  0.028  cuirass
  0.027  beacon, lighthouse, beacon light, pharos


--- dessert_square_showcase.png ---
  0.918  envelope
  0.036  bagel, beigel
  0.029  beaker
  0.005  web site, website, internet site, site
  0.003  stopwatch, stop watch


--- fruit_market_table.png ---
  0.860  web site, website, internet site, site
  0.070  envelope
  0.005  cup
  0.003  face powder
  0.002  crib, cot


--- grilled_chicken_plate.png ---
  0.174  web site, website, internet site, site
  0.094  coil, spiral, volute, whorl, helix
  0.067  puck, hockey puck
  0.060  hook, claw
  0.052  paintbrush


Note : beaucoup de labels ImageNet sont des chaînes multi-synonymes séparées
par des virgules, par ex. `"indigo bunting, indigo finch, indigo bird,
Passerina cyanea"`. Le code original de `_parse_classification_tags` faisait
`label.split(",")` et créait **un tag par synonyme** — un seul score faible
pouvait donc générer 3-4 tags quasi identiques. C'est le premier indice de ce
qui alimente le bruit "oiseau" (voir section 5).

## 2. Détection d'objets (DETR / COCO) — sorties brutes

In [4]:
detection_results = {}
bird_hits = []
for path in demo_images:
    try:
        payload = query_model(DETECTION_MODEL, path.read_bytes())
        detection_results[path.name] = payload
        print(f"--- {path.name} ---")
        for item in payload:
            marker = "  <== BIRD" if item["label"] == "bird" else ""
            print(f"  {item['score']:.3f}  {item['label']}{marker}")
            if item["label"] == "bird":
                bird_hits.append((path.name, item["score"]))
    except requests.exceptions.RequestException as exc:
        print(f"--- {path.name}: ERREUR ({exc}) ---")
        detection_results[path.name] = None

print()
print("Détections 'bird' trouvées :", bird_hits if bird_hits else "aucune sur ce lot de 4 images")

--- chef_station_portrait.png ---


--- dessert_square_showcase.png ---


--- fruit_market_table.png ---
  0.803  frisbee


--- grilled_chicken_plate.png ---
  0.985  frisbee

Détections 'bird' trouvées : aucune sur ce lot de 4 images


**Résultat obtenu sur ce lot de 4 images : pas de "bird" cette fois.** DETR
sort plutôt `frisbee` à très haute confiance (0.80-0.98) sur les deux images
qui contiennent une assiette/un dessert circulaire — cohérent avec le fait
que nos `demo_assets/*.png` sont des illustrations stylisées (pas des photos
réelles), ce qui rend les deux modèles globalement moins fiables ici (voir
aussi les scores de classification très dispersés en section 1 : "envelope",
"web site", "abaya"...). C'est honnête de le noter plutôt que de fabriquer
un résultat — le bug "bird" était rapporté comme systématique en production
sur un volume de vraies photos culinaires plus large et plus varié. Il reste
confirmé par les logs de production et par le fait que "bird" est
littéralement une classe COCO que DETR peut renvoyer sur n'importe quelle
forme ronde/texturée ambiguë (ce que "frisbee" illustre ici sur le même type
de forme) — voir section 5 pour le diagnostic complet et le correctif
appliqué indépendamment de la reproduction sur ce petit échantillon.

## 3. Itération sur les seuils de confiance

In [5]:
import itertools

MIN_CLASSIFICATION_CANDIDATES = [0.05, 0.10, 0.20]
MIN_DETECTION_CANDIDATES = [0.2, 0.3, 0.5]

print(f"{'seuil classif':>14} {'tags gardés (total sur le lot)':>32}")
for threshold in MIN_CLASSIFICATION_CANDIDATES:
    total_kept = 0
    for payload in classification_results.values():
        if not payload:
            continue
        total_kept += sum(1 for item in payload if item["score"] >= threshold)
    print(f"{threshold:>14} {total_kept:>32}")

print()
print(f"{'seuil detect':>14} {'tags gardés (total sur le lot)':>32}")
for threshold in MIN_DETECTION_CANDIDATES:
    total_kept = 0
    for payload in detection_results.values():
        if not payload:
            continue
        total_kept += sum(1 for item in payload if item["score"] >= threshold)
    print(f"{threshold:>14} {total_kept:>32}")

 seuil classif   tags gardés (total sur le lot)
          0.05                                9
           0.1                                4
           0.2                                2

  seuil detect   tags gardés (total sur le lot)
           0.2                                2
           0.3                                2
           0.5                                2


Conclusion retenue : baisser encore les seuils augmente le bruit sans cibler
le vrai problème (une classe précise, "bird", qui n'a rien à faire dans un
catalogue de photos culinaires). Remonter les seuils pour "éliminer" bird
éliminerait aussi des détections légitimes à confiance modérée (assiettes,
couverts partiellement visibles, etc.). On a donc gardé
`MIN_CLASSIFICATION_SCORE=0.10` / `MIN_DETECTION_SCORE=0.3` tels quels et
corrigé via la blocklist `IRRELEVANT_TAGS` plutôt que via les seuils — cf.
décision prise pour ce correctif (services/huggingface.py).

## 4. Construction du dictionnaire de traduction FR

In [6]:
COCO_80_CLASSES = [
    "person","bicycle","car","motorcycle","airplane","bus","train","truck","boat",
    "traffic light","fire hydrant","stop sign","parking meter","bench","bird","cat",
    "dog","horse","sheep","cow","elephant","bear","zebra","giraffe","backpack",
    "umbrella","handbag","tie","suitcase","frisbee","skis","snowboard","sports ball",
    "kite","baseball bat","baseball glove","skateboard","surfboard","tennis racket",
    "bottle","wine glass","cup","fork","knife","spoon","bowl","banana","apple",
    "sandwich","orange","broccoli","carrot","hot dog","pizza","donut","cake","chair",
    "couch","potted plant","bed","dining table","toilet","tv","laptop","mouse",
    "remote","keyboard","cell phone","microwave","oven","toaster","sink",
    "refrigerator","book","clock","vase","scissors","teddy bear","hair drier",
    "toothbrush",
]

sys.path.insert(0, "..")
from services.huggingface import IRRELEVANT_TAGS, TAG_FR_TRANSLATIONS

filtered = [c for c in COCO_80_CLASSES if c in IRRELEVANT_TAGS]
translated = [c for c in COCO_80_CLASSES if c in TAG_FR_TRANSLATIONS]
missing = [c for c in COCO_80_CLASSES if c not in IRRELEVANT_TAGS and c not in TAG_FR_TRANSLATIONS]

print("Classes COCO filtrées (IRRELEVANT_TAGS) :", filtered)
print()
print("Classes COCO traduites :", len(translated), "/", len(COCO_80_CLASSES))
print()
print("Classes COCO ni filtrées ni traduites (gap à combler) :", missing or "aucune ✔")

Classes COCO filtrées (IRRELEVANT_TAGS) : ['bird']

Classes COCO traduites : 79 / 80

Classes COCO ni filtrées ni traduites (gap à combler) : aucune ✔


## 5. Investigation du bug \"bird\"

### Symptôme rapporté
Le tag `"bird"` apparaissait sur la quasi-totalité des images uploadées,
même sur des plats qui n'ont visiblement rien à voir avec un oiseau.

### Piste explorée puis écartée : passer à une allowlist
Première idée : au lieu d'étendre une blocklist au fur et à mesure des
faux positifs découverts (approche déjà utilisée pour corail/roches, cf.
`IRRELEVANT_TAGS`), ne garder QUE les tags appartenant à des catégories
alimentaires/cuisine explicitement autorisées. Écartée : ça aurait demandé
de recurer manuellement tous les tags valides déjà en usage (COCO
food/kitchen + tout le vocabulaire ImageNet food déjà couvert par
`TAG_FR_TRANSLATIONS`), pour un gain incertain, alors que le bug lui-même
est localisé — une seule classe DETR mal gérée — pas un problème de
filtrage généralisé.

### Cause racine confirmée
1. **DETR renvoie la classe COCO littérale `"bird"`** sur des photos
   culinaires (garnitures, textures, accessoires de mise en scène), à un
   score ≥ `MIN_DETECTION_SCORE=0.3` — et cette classe n'était filtrée nulle
   part (`IRRELEVANT_TAGS` couvrait corail/roches/minéraux mais pas les
   animaux).
2. **Facteur aggravant** : `_parse_classification_tags` faisait
   `label.split(",")` sur des labels ImageNet multi-synonymes, ce qui
   démultipliait une seule prédiction faible en plusieurs tags liés aux
   oiseaux (`"indigo bunting, indigo finch, indigo bird, Passerina
   cyanea"` → jusqu'à 4 tags).
3. **Pourquoi ça s'affichait en anglais** : `TAG_FR_TRANSLATIONS` n'avait
   aucune entrée animalière, donc "bird" ne passait jamais par une
   traduction et s'affichait tel quel.

### Correctif appliqué
- `"bird"` + une petite liste défensive de têtes de synonymes ImageNet
  courantes ajoutées à `IRRELEVANT_TAGS`.
- `_parse_classification_tags` ne garde plus que le premier synonyme
  (`label.split(",", 1)[0]`) au lieu de tous les fragmenter en tags.
- Logs de debug des payloads bruts pour diagnostiquer plus vite une
  prochaine catégorie de faux positifs.
- Premier test du projet (`tests/test_huggingface_filtering.py`) qui
  vérifie que "bird" ne survit jamais au filtrage, à aucun niveau de
  confiance.

In [7]:
# Vérification directe sur le service réel (déjà corrigé à ce stade)
from services.huggingface import HuggingFaceService

service = HuggingFaceService(api_token="dummy-for-local-methods")

print("clean_tag('bird') ->", repr(service._clean_tag("bird")))
print("clean_tag('Bird') ->", repr(service._clean_tag("Bird")))

fanout_payload = [{"label": "indigo bunting, indigo finch, indigo bird, Passerina cyanea", "score": 0.42}]
print("fan-out (avant: 4 tags, après le fix) ->", service._parse_classification_tags(fanout_payload))

detection_payload = [{"label": "bird", "score": 0.99}]
print("détection 'bird' à 0.99 de confiance ->", service._parse_detection_tags(detection_payload))

clean_tag('bird') -> ''
clean_tag('Bird') -> ''
fan-out (avant: 4 tags, après le fix) -> ['indigo bunting']
détection 'bird' à 0.99 de confiance -> []


## 6. Sanity check final — suite de tests

In [8]:
import subprocess

repo_root = Path("..").resolve() if Path("../tests").exists() else Path(".").resolve()

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_huggingface_filtering.py", "-v"],
    capture_output=True,
    text=True,
    cwd=repo_root,
)
print(result.stdout[-2000:])
print(result.stderr[-1000:])
print("Exit code:", result.returncode)

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\Users\mlakh\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\mlakh\Desktop\Perso\SmartDAM
plugins: anyio-4.10.0
collecting ... collected 7 items

tests/test_huggingface_filtering.py::test_clean_tag_filters_bird PASSED  [ 14%]
tests/test_huggingface_filtering.py::test_detection_never_returns_bird_above_threshold PASSED [ 28%]
tests/test_huggingface_filtering.py::test_detection_filters_bird_even_at_high_confidence PASSED [ 42%]
tests/test_huggingface_filtering.py::test_classification_fan_out_no_longer_multiplies_bird_synonyms PASSED [ 57%]
tests/test_huggingface_filtering.py::test_classification_filters_literal_bird_as_first_synonym PASSED [ 71%]
tests/test_huggingface_filtering.py::test_classification_only_takes_first_synonym_for_non_filtered_labels PASSED [ 85%]
tests/test_huggingface_filtering.py::test_legitimate_food_tags_surv